In [1]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2
import pickle

In [2]:
df = pd.read_excel(r"C:\web dev\tourist\Book2 all csv data.xlsx")


In [3]:
# Clean coordinates column
def parse_coords(coord):
    try:
        coord = coord.replace("Coordinates:", "").replace("° N", "").replace("° E", "")
        parts = coord.split(",")
        if len(parts) == 2:
            lat = float(parts[0].strip())
            lon = float(parts[1].strip())
            return lat, lon
        else:
            return None, None
    except:
        return None, None


In [4]:
df[['lat', 'lon']] = df['Coordinates'].apply(lambda x: pd.Series(parse_coords(x)))

In [5]:
# Haversine distance function
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Radius of Earth in km
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

In [6]:
# Function to recommend 5 nearby places given a city
def recommend_places(city_name):
    city_row = df[df['Destination'].str.lower() == city_name.lower()]
    if city_row.empty:
        return f"No data found for city: {city_name}"
    
    lat1, lon1 = city_row.iloc[0][['lat', 'lon']]
    
    df['distance'] = df.apply(lambda row: haversine(lat1, lon1, row['lat'], row['lon']), axis=1)
    nearby = df[df['Destination'].str.lower() != city_name.lower()].sort_values('distance').head(5)
    return nearby[['Destination', 'Category', 'Decription', 'distance']]

In [7]:
# Example
print(recommend_places("Tirupati"))

                         Destination             Category  \
11  Sri Venkateswara Zoological Park   Category: Wildlife   
43                         Kanipakam  Category: Religious   
20               Srikalahasti Temple  Category: Religious   
28                        Tada Falls  Category: Waterfall   
45                Talakona Waterfall  Category: Waterfall   

                                           Decription   distance  
11  Description: Home to diverse flora and fauna, ...   1.616653  
43  Description: Famous for the Kanipakam Vinayaka...  21.429772  
20  Description: Famous for its ancient architectu...  33.492771  
28  Description: Trek through lush forests to reac...  34.356933  
45  Description: Highest waterfall in Andhra Prade...  46.035787  


In [8]:
# Save the cleaned dataset as pickle
with open("places_dataset.pkl", "wb") as f:
    pickle.dump(df, f)

print("✅ Cleaned dataset saved as places_dataset.pkl")

✅ Cleaned dataset saved as places_dataset.pkl
